In [ ]:
'''
Question 1) List two benefits of using Mongoose instead of the native MongoDB driver. Also mention one scenario where using the native driver could be preferable.
Answer: Two Benefits of Using Mongoose:
	Schema Validation & Typing: Mongoose enforces built-in and custom schema validation rules (e.g., required, minlength, data types) before data reaches the database, catching invalid data early.

	Middleware & Business Logic Hooks: It provides pre- and post-hooks (e.g., pre('save'), post('remove')), making it easy to automate tasks like password hashing, timestamp updates, or cascading deletes.

	Scenario Where Native Driver is Preferable:  High-Performance / Bulk Operations: When running ultra-fast, high-throughput bulk reads/writes or complex aggregation pipelines where object mapping adds unnecessary CPU overhead. The native driver bypasses schema casting and document wrapping, yielding lower memory overhead and better raw speed.



Question 2) Create a Mongoose schema for a `User` with properties:
- `name` (required string, minimum length 2, maximum length 50)
- `email` (required string)

Then export a `User` model.

Answer: const mongoose = require('mongoose');

// Define Schema
const userSchema = new mongoose.Schema({
  name: {
    type: String,
    required: true,
    minlength: 2,
    maxlength: 50
  },
  email: {
    type: String,
    required: true
  }
});

// Create and Export Model
const User = mongoose.model('User', userSchema);
module.exports = User;




Question 3) Write code to (a) fetch all users, (b) fetch a user by ID, and (c) add a query helper byEmailDomain(domain) that returns users whose email ends with the given domain. Show example usage.
Answer:
const mongoose = require('mongoose');

// Schema Definition with Query Helper
const userSchema = new mongoose.Schema({
  name: String,
  email: String
});

// (c) Adding query helper 'byEmailDomain'
userSchema.query.byEmailDomain = function(domain) {
  return this.where({ email: new RegExp(`@${domain}$`, 'i') });
};

const User = mongoose.model('User', userSchema);

async function examples() {
  // (a) Fetch all users
  const allUsers = await User.find({});

  // (b) Fetch a user by ID
  const userId = '60d5ecb8b3b3a32f84288350';
  const userById = await User.findById(userId);

  // Example usage of (c) Query Helper
  const gmailUsers = await User.find().byEmailDomain('gmail.com');
  console.log(gmailUsers);
}

Question 4) Demonstrate two ways to change a user’s name to "Rita" using Mongoose: (1) load, modify, then save(), and (2) use findOneAndUpdate(). Mention when save() is preferable.
Answer: 1. Load, Modify, and save()
	const user = await User.findById(userId);
if (user) {
  user.name = "Rita";
  await user.save();
}
2. Using findOneAndUpdate()
JavaScript
const updatedUser = await User.findOneAndUpdate(
  { _id: userId },
  { name: "Rita" },
  { new: true, runValidators: true } // Returns updated doc & runs validations
);

When save() is Preferable:
When you need Mongoose schema validation hooks (e.g., pre('save') for password hashing) to execute.
When updating deeply nested properties or manipulating complex instance methods prior to persisting data.
When dealing with concurrent changes using Mongoose's built-in optimistic concurrency control (__v version key).


Question 5) Explain when you would embed a subdocument versus using references in MongoDB/Mongoose. Show code for a one-to-many reference (e.g. a Post having many Comment references).
Answer:
Embed Subdocuments: Use when data is tightly coupled, bounded/small in size, and usually retrieved together (e.g., address details on a user profile). It optimizes performance by reducing database roundtrips.

Use References: Use when data is unbounded (can grow endlessly), shared across multiple entities, or rarely accessed alongside the parent document (e.g., millions of user comments or logs).
const mongoose = require('mongoose');

// 1. Comment Schema
const commentSchema = new mongoose.Schema({
  text: String,
  createdAt: { type: Date, default: Date.now }
});
const Comment = mongoose.model('Comment', commentSchema);

// 2. Post Schema with references to Comment
const postSchema = new mongoose.Schema({
  title: String,
  content: String,
  comments: [{
    type: mongoose.Schema.Types.ObjectId,
    ref: 'Comment'
  }]
});
const Post = mongoose.model('Post', postSchema);

// Example usage populating comments
async function getPostWithComments(postId) {
  const post = await Post.findById(postId).populate('comments');
  console.log(post);
}


Question 6) Write a Mongoose (or MongoDB) aggregation pipeline to group users by email domain and count how many users per domain. Then show how you could use $lookup to join a bounces collection (with bounce counts per domain).
Answer:
const mongoose = require('mongoose');
const User = mongoose.model('User');

async function getDomainStats() {
  const result = await User.aggregate([
    // 1. Extract domain from email (split by '@') and group to count users
    {
      $group: {
        _id: { $arrayElemAt: [{ $split: ['$email', '@'] }, 1] },
        userCount: { $sum: 1 }
      }
    },
    // 2. Join with the 'bounces' collection on domain name
    {
      $lookup: {
        from: 'bounces',         // Target collection name
        localField: '_id',        // Domain extracted from user email
        foreignField: 'domain',   // Field in bounces collection
        as: 'bounceData'          // Array field containing matched bounces
      }
    }
  ]);

  return result;
}

Question 7) Show minimal code (ESM) to connect to MongoDB Atlas using Mongoose. Assume the URI is stored in the environment variable MONGODB_URI.
Answer:
import mongoose from 'mongoose';

const connectDB = async () => {
  try {
    await mongoose.connect(process.env.MONGODB_URI);
    console.log('MongoDB Atlas connected successfully.');
  } catch (error) {
    console.error('Connection failed:', error.message);
    process.exit(1);
  }
};

connectDB();


Question 8) Give a short description of Express. Then write routes for:
    • GET /health → responds “OK”
    • GET /users/:id → responds with { id: <id> }
    • GET /search?term=... → responds with { term: <term> }
Answer: Express is a minimal, unopinionated, and fast web framework for Node.js. It simplifies routing, middleware integration, and HTTP request/response handling when building web applications and RESTful APIs.
import express from 'express';

const app = express();

// GET /health → responds "OK"
app.get('/health', (req, res) => {
  res.send('OK');
});

// GET /users/:id → responds with { id: <id> }
app.get('/users/:id', (req, res) => {
  res.json({ id: req.params.id });
});

// GET /search?term=... → responds with { term: <term> }
app.get('/search', (req, res) => {
  res.json({ term: req.query.term || '' });
});

app.listen(3000);


Question 9) Create a router for /api/users with two endpoints: GET / returning empty array and POST / returning the posted JSON. Also add a logger middleware that prints method and URL for every request.
Answer:
import express from 'express';

const app = express();
app.use(express.json()); // Middleware to parse JSON body

// Logger middleware
app.use((req, res, next) => {
  console.log(`${req.method} ${req.url}`);
  next();
});

// Define User Router
const userRouter = express.Router();

// GET /api/users
userRouter.get('/', (req, res) => {
  res.json([]);
});

// POST /api/users
userRouter.post('/', (req, res) => {
  res.json(req.body);
});

// Mount the router
app.use('/api/users', userRouter);

app.listen(3000);

Question 10) Add an error-handling middleware to catch thrown errors and respond with { error: <message> }, status 500. Demonstrate by a route that throws an error.
Answer: import express from 'express';

const app = express();

// Route that throws an error
app.get('/cause-error', (req, res, next) => {
  try {
    throw new Error('Something went wrong on the server!');
  } catch (error) {
    next(error); // Pass error to error handler
  }
});

// Error-handling middleware (must have 4 parameters: err, req, res, next)
app.use((err, req, res, next) => {
  console.error(err.stack);
  res.status(500).json({ error: err.message || 'Internal Server Error' });
});

app.listen(3000);

Question 11) Write middleware auth() that checks for Authorization: Bearer <token> header, verifies the token, and sets req.user. Then protect route GET /me to return user info if valid, otherwise 401.
Answer:
import express from 'express';
import jwt from 'jsonwebtoken';

const app = express();
const JWT_SECRET = process.env.JWT_SECRET || 'your_secret_key';

// Auth Middleware
const auth = (req, res, next) => {
  const authHeader = req.headers['authorization'];
  const token = authHeader && authHeader.split(' ')[1]; // Extract token after "Bearer "

  if (!token) {
    return res.status(401).json({ error: 'Access denied. No token provided.' });
  }

  try {
    const decoded = jwt.verify(token, JWT_SECRET);
    req.user = decoded; // Attach payload to request object
    next();
  } catch (err) {
    return res.status(401).json({ error: 'Invalid or expired token.' });
  }
};

// Protected GET /me route
app.get('/me', auth, (req, res) => {
  res.json({ user: req.user });
});

app.listen(3000);



'''